## Classes, Errors, Modules, File IO


## Classes

In [1]:
class Circle(object):  # object is super class, similar to "Object" in Java
                       # can be omitted, if not inheriting from another (super)class
    pi = 3.142  # class variable
                # similar to "static" in Java
    
    def __init__(self, radius):
        self.radius = radius  # self: similar to "this" in Java
                              # self.radius is an instance variable
                              # instance variable introduced within __init__()

    def area(self):
        return self.pi * self.radius * self.radius


In [2]:
c = Circle(3)  # creates an instance
c.area()  # notice you don't have to pass "self" as an argument

28.278

Why did we not have to add ``self`` when calling ``area()``?

When we do ``c.area()``, Python converts it in the background to ``Circle.area(c)``

## Coding style conventions

- Class name: camel case, e.g. ``Circle``&ensp;``NiceCircle``&ensp;``ReallyNiceCircle``

- Variable name for instance: lowercase with underscores, e.g. ``my_circle``&ensp;``your_really_nice_circle``

- Method name: lower case with underscores, e.g. ``area()``&ensp;``half_of_area()``

## "Dunder" methods

- "Dunder" means double underscore

- Officially called magic methods or special methods
  - ``__init__()``
  - ``__str__()``
  - ``__repr__()``
  - ``__call__()``

In [3]:
class Cat(object):
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __str__(self):
        return f'Cat named {self.name} with {self.color} hair'

    def __repr__(self):
        return f'class=Cat, name={self.name}, color={self.color}'

    def __call__(self, some_input):
        return f'Meaow {some_input}'


In [4]:
marble = Cat('Marble', 'gray')
print(marble)

Cat named Marble with gray hair


In [10]:
# invokes __call__ and looks like a function call
marble('XXX')

'Meaow XXX'

In [11]:
str(marble)

'Cat named Marble with gray hair'

In [12]:
marble

class=Cat, name=Marble, color=gray

In [13]:
repr(marble)

'class=Cat, name=Marble, color=gray'

Don't the two methods do the same thing?

- Rule of thumb: ``__str__()`` is user-friendly for customers, ``__repr__()`` is technical for programmers

- When ``__str__()`` is missing, ``__repr__()`` will step in

- **Conclusion:** Every class should have ``__repr__()``!

Rule of thumb: ``__str__()`` is user-friendly for customers, ``__repr__()`` is technical for programmers

In [14]:
import datetime
today = datetime.date.today()

str(today)

'2026-04-24'

In [15]:
repr(today)

'datetime.date(2026, 4, 24)'

It's good practice to include the class name in ``__repr__()``

When ``__str__()`` is missing, ``__repr__()`` will step in

## "Dunder" variables

Depending on the context (module, class, method/function), the same dunder variable name stores different things. See https://www.pythonmorsels.com/dunder-variables/ for more details.

### ``__name__``

In [13]:
Cat.__name__  # context = class

'Cat'

### ``__class__``

In [14]:
marble.__class__  # context = instance

__main__.Cat

``__main__`` is the name of the top-level code environment

### ``__dict__``

In [15]:
marble.__dict__  # context = instance

{'name': 'Marble', 'color': 'gray'}

``__dict__`` yields the attributes

## ``is`` vs. ``==``

``is`` checks if two variables point to the same instance, similar to ``==`` in Java

In [16]:
marble2 = marble
marble is marble2

True

``==`` checks if two instances have the same content, similar to ``equals()`` in Java

- Requires implementing ``__eq__()``, because ``==`` invokes ``__eq__()``

- Starting in Python 3, it is no longer obligatory to also implement ``__ne__()`` (for not equal)

In [52]:
class Cat(object):
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __eq__(self, other):  ##### new method
        return self.name == other.name  # same name is enough, color does not matter

    def __str__(self):
        return f'Cat named {self.name} with {self.color} hair'

    def __repr__(self):
        return f'class=Cat, name={self.name}, color={self.color}'


In [53]:
marble = Cat('Marble', 'gray')
marble2 = Cat('Marble', 'black')

marble == marble2

True

In [54]:
marble is marble2

False

## Class variables vs. instance variables

- A class variable has the SAME value for every instance of the same class, think of "static" in Java

- An instance variable has DIFFERENT values for different instances of the same class

In [20]:
class Dog:
    num_legs = 4  # class variable

    def __init__(self, name):
        self.name = name  # instance variable


In [21]:
benny = Dog('Benny')
bonny = Dog('Bonny')

benny.num_legs  # instance variable

4

In [22]:
bonny.num_legs  # instance variable

4

In [23]:
Dog.num_legs  # class variable

4

## Instance methods vs. class methods vs. static methods

In [17]:
class ClassWithMethods:
    def instance_method(self):
        return 'instance method', self

    @classmethod  # this is a decorator, similar to annotation in Java
    def class_method(cls):  # fun fact, "cls" is a swear word in Cantonese >:D
        return 'class method', cls

    @staticmethod
    def static_method():  # neither self nor cls is passed as parameter
        return 'static method'


## Instance methods

Similar to plain methods in a Java class

In [18]:
instance = ClassWithMethods()
instance.instance_method()

('instance method', <__main__.ClassWithMethods at 0x7fe279525520>)

In [19]:
# In the background, this was actually done:
ClassWithMethods.instance_method(instance)

('instance method', <__main__.ClassWithMethods at 0x7fe279525520>)

## Class methods

Similar to "static" methods in a Java class

In [20]:
ClassWithMethods.class_method()

('class method', __main__.ClassWithMethods)

Notice that the class is passed automatically as a parameter

In fact, we can also call the class method via an instance

In [21]:
instance.class_method()

('class method', __main__.ClassWithMethods)

What are class methods good for?

- Class polymorphism -- in Python, there is polymorphism at the instance AND at the class levels
- We won't get into further details here

## Static methods

No equivalent in Java

In [22]:
ClassWithMethods.static_method()  # can call static method via class

'static method'

In [23]:
instance.static_method()  # can also call static method via instance

'static method'

What are static methods good for?

- Most common use is to "park" a utility method within the namespace of a class
- So do this sparingly!

Example:

In [24]:
class StuffedCrustPizza:
    @staticmethod
    def calculate_cheese_needed(radius):  # radius of pizza
        # as a static method, it cannot access anything like self.radius
        # so the radius must be explicitly passed in
        cheese = 2 * math.pi * (radius-1) * ...


## Public vs. "private" attributes

- There is no genuine private attributes, everything is basically public in Python

- According to convention, a variable that starts with an underscore is "private"

In [26]:
class GotSecret():
    def __init__(self):
        self._secret = 'Cats like dogs'  # Bad manners if you access this "private" variable

secret = GotSecret()
secret._secret  # But no one can stop you from doing this

'Cats like dogs'

# Errors

Bugs come in three basic flavours:

- *Syntax errors:* 
    - Code is not valid Python (easy to fix, except for some whitespace things)
    
- *Runtime errors:* 
    - Syntactically valid code fails, often because variables contain wrong values

- *Logical errors:* 
    - Code executes successfully, but the result is wrong (difficult to fix)

# What happened to compile time errors?

- Compile time errors = syntax correct, but compiler does not understand the code, e.g. undefined variable

- In principle, compile time errors still exists, but...

- Because Python is interpreted, such errors only shows up when you run the code

- So we can loosely call these errors runtime errors also

## Runtime Errors

### Trying to access undefined variables

In [1]:
# Q was never defined
print(Q)

NameError: name 'Q' is not defined

### Trying to execute unsupported operations

In [2]:
1 + 'abc'

TypeError: unsupported operand type(s) for +: 'int' and 'str'

### Trying to access elements in collections that don't exist

In [3]:
L = [1, 2, 3]
L[1000]

IndexError: list index out of range

### Trying to compute a mathematically ill-defined result

In [4]:
2 / 0

ZeroDivisionError: division by zero

## Catching Exceptions: ``try`` and ``except``


In [5]:
try:
    print("this gets executed first")
except:
    print("this gets executed only if there is an error")

this gets executed first


In [6]:
try:
    print("let's try something:")
    x = 1 / 0  # ZeroDivisionError
    print("we never reach this line")
except:
    print("something bad happened!")

let's try something:
something bad happened!


In [7]:
def safe_divide(a, b):
    """
    A function that does a division and returns a half-sensible 
    value even for mathematically ill-defined results
    """
    try:
        return a / b
    except:
        return 1E100

In [8]:
print(safe_divide(1, 2))
print(safe_divide(1, 0))

0.5
1e+100


### What about errors that we didn't expect?


In [9]:
safe_divide (1, '2')

1e+100

### It's good practice to always catch errors explicitly:

All other errors will be raised as if there were no try/except clause.


In [10]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return 1E100

In [11]:
safe_divide(1, '2')

TypeError: unsupported operand type(s) for /: 'int' and 'str'

## Throwing Errors

- When your code is executed, make sure that it's clear what went wrong in case of errors.
- Throw [specific errors built into Python](https://docs.python.org/3/tutorial/errors.html)
- Write your own error classes

In [12]:
raise RuntimeError("my error message")

RuntimeError: my error message

## Specific Errors

In [13]:
def safe_divide(a, b):
    if (not issubclass(type(a), float)) or (not issubclass(type(b), float)):
        raise ValueError("Arguments must be floats")
    try:
        return a / b
    except ZeroDivisionError:
        return 1E100

In [14]:
safe_divide(1, '2')

ValueError: Arguments must be floats

## Accessing Error Details

In [15]:
import warnings

def safe_divide(a, b):
    if (not issubclass(type(a), float)) or (not issubclass(type(b), float)):
        raise ValueError("Arguments must be floats")
    try:
        return a / b
    except ZeroDivisionError as err:
        warnings.warn("Caught Error {} with message '{}'".format(type(err),err) + 
                " - will just return a large number instead")
        return 1E100
    
    

In [16]:
safe_divide(1., 0.)

/var/folders/fb/nnrbj09j32jdr_jvd_fnmyn00000gn/T/ipykernel_72083/1926164315.py:9: UserWarning: Caught Error <class 'ZeroDivisionError'> with message 'float division by zero' - will just return a large number instead
  warnings.warn("Caught Error {} with message '{}'".format(type(err),err) +


1e+100

## Built-in errors in Python and their hierarchy

- See https://docs.python.org/3/library/exceptions.html

# Modules

> Python modules are files containing Python code (functions, classes, variables) that can be imported and reused in other programs
* Reusable Code
* Organization / cleaner code base structure
* Namespace Separation preventing naming conflicts


## Modules vs Packages

> Module: A single .py file containing Python code (functions, classes, variables). Example: math.py.

> Package: A directory containing multiple modules (and an ``__init__.py`` file to mark it as a package). Packages help organize modules into hierarchies.
Analogy:

A module is like a single book (e.g., book.py).
A package is like a bookshelf (a folder) holding multiple books (modules).

So, all packages contain modules, but not all modules are packages.



## Loading Modules: the ``import`` Statement

- Explicit imports (best)
- Explicit imports with alias (ok for long package names)
- Explicit import of module contents
- Implicit imports (to be avoided)

## Creating Modules

- Create a file called [somefilename].py
- In a (i)python shell change dir to that containing dir
- type 

```python
import [somefilename]
```

Now all classes, functions and variables in the top level namespace (more on this a few slides later) are available. 

Let's assume we have a file `mymodule.py` in the current working directory with the content:

```python
mystring = 'hello world'

def myfunc():
    print(mystring)
```

In [17]:
import mymodule
mymodule.mystring

'hello world'

In [18]:
mymodule.myfunc()

hello world


## Explicit module import

Explicit import of a module preserves the module's content in a namespace.

In [19]:
import math
math.cos(math.pi)  # cos exists in math's namespace

-1.0

## Explicit module import with aliases

For longer module names, it's not convenient to use the full module name. 

In [20]:
import numpy as np
np.cos(np.pi)

-1.0

## Explicit import of module contents
You can import specific elements separately. 

In [21]:
from math import cos, pi
cos(pi)  # cos, pi now exist in global namespace

-1.0

## Implicit import of module contents
You can import all elements of a module into the global namespace. Use with caution.

In [22]:
cos = 0
from math import *
sin(pi) ** 2 + cos(pi) ** 2

1.0

## Namespaces

- A namespace is similar to a map, where a name points to an object

- There are built-in, global, local, and enclosing namespaces

- In the built-in namespace are names like ``AssertionError``, ``float``, ``__name__``, etc.

- In the global namespace are names defined in the "main" program, including names coming from ``import``

- Local namespace is mostly about functions

- Enclosing namespace is about nested (!) functions

For more details, see https://realpython.com/python-namespaces-scope/

# File IO and Encoding


- Files are opened with ``open``
- By default in ``'r'`` mode, reading text mode, line-by-line
- ``open`` chooses an appropriate default encoding based on the environment / platform, but you can also specify enconding with e.g. ``encoding='utf-8'``

## Reading Text


In [23]:
path = 'umlauts.txt'
f = open(path, encoding='utf-8')
lines = [x.strip() for x in f]
f.close()
lines

['Eichhörnchen', 'Flußpferd', '', 'Löwe', '', 'Eichelhäher']

In [24]:
# for easier cleanup
with open(path, encoding='utf-8') as f:
    lines = [x.rstrip() for x in f]
lines

['Eichhörnchen', 'Flußpferd', '', 'Löwe', '', 'Eichelhäher']

## Detour: Context Managers

Often, like when opening files, you want to make sure that the file handle gets closed in any case.

```python
file = open(path, 'w')
try:
    # an error
    1 / 0
finally:
    file.close()
```

Context managers are a convenient shortcut:
```python
with open(path, 'w') as opened_file:
    # an error
    1/0
```

## Writing Text


In [25]:
with open('tmp.txt', 'w', encoding='utf-8') as handle:
    handle.writelines(x for x in open(path, encoding='utf-8') if len(x) > 1)
[x.rstrip() for x in open('tmp.txt', encoding='utf-8')]

['Eichhörnchen', 'Flußpferd', 'Löwe', 'Eichelhäher']

## Reading Bytes


In [26]:
# remember 't' was for text reading/writing
with open(path, 'rt', encoding='utf-8') as f:
    # just the first 6 characters
    chars = f.read(6)
chars

'Eichhö'

In [27]:
# now we read the file content as bytes
with open(path, 'rb') as f:  # no encoding='utf-8' since we want the raw bytes
    # just the first 6 bytes
    data = f.read(6)

In [28]:
# byte representation
data.decode('utf8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 5: unexpected end of data

In [29]:
# decoding error, utf-8 has variable length character encodings
data[:4].decode('utf-8')

'Eich'